# 11 — Hierarchical Agents

**Model:** DeepSeek-V3 via Nebius AI Studio
**Pattern:** 3-Tier Strategic → Tactical → Execution Architecture
**Difficulty:** ⭐⭐⭐⭐⭐

---

## The Problem

As tasks grow more complex, flat multi-agent systems start to break down. The supervisor gets overloaded managing too many direct reports.

**Hierarchical Agent Systems** mirror how human organizations work:
- **Strategic layer**: Sets the overall goal, allocates responsibilities
- **Tactical layer**: Breaks department goals into actionable tasks, coordinates workers
- **Execution layer**: Executes specific tasks with tools

Each layer only communicates with adjacent layers — reducing complexity.

## Architecture

```
             [Strategic Layer: Goal Decomposer]
          Sets research + content department objectives
                 │                    │
    [Research Manager]          [Content Manager]
     Assigns search tasks        Assigns writing tasks
         │         │                 │          │
    [Searcher] [Fact Check]    [Outliner]   [Writer]
```

## Setup

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_nebius import ChatNebius
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Annotated, Optional
import operator
from tavily import TavilyClient
from pydantic import BaseModel

llm = ChatNebius(
    model="deepseek-ai/DeepSeek-V3",
    temperature=0,
    api_key=os.environ["NEBIUS_API_KEY"]
)
tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
print("Setup complete.")

## State

In [ ]:
class HierarchyState(TypedDict):
    goal: str
    research_objective: str
    content_objective: str
    research_tasks: List[str]
    content_tasks: List[str]
    research_results: Annotated[List[str], operator.add]
    content_pieces: Annotated[List[str], operator.add]
    final_output: Optional[str]

## Strategic Layer

In [ ]:
class StrategicPlan(BaseModel):
    research_objective: str
    content_objective: str

strategic_llm = llm.with_structured_output(StrategicPlan)

def strategic_node(state: HierarchyState) -> dict:
    print(f"\n[STRATEGIC] Decomposing goal: '{state['goal'][:80]}...'")
    plan: StrategicPlan = strategic_llm.invoke([
        SystemMessage(content=(
            "You are a strategic planner. Given a high-level goal, define two department objectives:\n"
            "1. Research objective: what facts/data needs to be gathered\n"
            "2. Content objective: what the final deliverable should be and its format\n"
            "Be specific and measurable."
        )),
        HumanMessage(content=f"Goal: {state['goal']}")
    ])
    print(f"[STRATEGIC] Research: {plan.research_objective[:80]}")
    print(f"[STRATEGIC] Content: {plan.content_objective[:80]}")
    return {
        "research_objective": plan.research_objective,
        "content_objective": plan.content_objective,
        "research_results": [], "content_pieces": []
    }

## Tactical Layer

In [ ]:
def research_manager_node(state: HierarchyState) -> dict:
    print(f"\n[TACTICAL:Research] Planning research tasks...")
    response = llm.invoke([
        SystemMessage(content=(
            "You manage a research team. Break the research objective into 2-3 specific "
            "web search queries. Return as a numbered list."
        )),
        HumanMessage(content=f"Research objective: {state['research_objective']}")
    ])
    lines = [l.strip().lstrip('123456789.)-').strip()
             for l in response.content.split('\n') if l.strip() and l.strip()[0].isdigit()]
    tasks = lines[:3] if lines else [state["research_objective"]]
    print(f"[TACTICAL:Research] {len(tasks)} tasks assigned.")
    return {"research_tasks": tasks}


def content_manager_node(state: HierarchyState) -> dict:
    print(f"\n[TACTICAL:Content] Planning content tasks...")
    response = llm.invoke([
        SystemMessage(content=(
            "You manage a content team. Break the content objective into 2-3 specific "
            "writing tasks. Return as a numbered list."
        )),
        HumanMessage(content=f"Content objective: {state['content_objective']}")
    ])
    lines = [l.strip().lstrip('123456789.)-').strip()
             for l in response.content.split('\n') if l.strip() and l.strip()[0].isdigit()]
    tasks = lines[:3] if lines else [state["content_objective"]]
    print(f"[TACTICAL:Content] {len(tasks)} tasks assigned.")
    return {"content_tasks": tasks}

## Execution Layer

In [ ]:
def research_execution_node(state: HierarchyState) -> dict:
    print(f"\n[EXECUTION:Research] Running {len(state['research_tasks'])} searches...")
    results = []
    for task in state["research_tasks"]:
        print(f"  Searching: {task[:60]}...")
        try:
            sr = tavily.search(query=task, max_results=2)
            snippets = [r["content"] for r in sr.get("results", [])]
            result = f"[Search: {task}]\n" + "\n".join(snippets)
        except Exception as e:
            result = f"[Search failed: {task}] Error: {e}"
        results.append(result)
    return {"research_results": results}


def content_execution_node(state: HierarchyState) -> dict:
    print(f"\n[EXECUTION:Content] Writing {len(state['content_tasks'])} content pieces...")
    research_context = "\n\n".join(state["research_results"])
    pieces = []
    for task in state["content_tasks"]:
        print(f"  Writing: {task[:60]}...")
        response = llm.invoke([
            SystemMessage(content="You are a professional writer. Use the research provided."),
            HumanMessage(content=(
                f"Research:\n{research_context}\n\n"
                f"Goal: {state['goal']}\n\nWriting task: {task}"
            ))
        ])
        pieces.append(f"[{task}]\n{response.content}")
    return {"content_pieces": pieces}


def final_assembly_node(state: HierarchyState) -> dict:
    print("\n[STRATEGIC] Assembling final output...")
    all_pieces = "\n\n".join(state["content_pieces"])
    response = llm.invoke([
        SystemMessage(content="You are an editor. Assemble content pieces into a coherent final document."),
        HumanMessage(content=(
            f"Goal: {state['goal']}\n\nContent pieces:\n{all_pieces}\n\n"
            "Produce the final polished document:"
        ))
    ])
    return {"final_output": response.content}

## Build the Graph

In [ ]:
builder = StateGraph(HierarchyState)
builder.add_node("strategic", strategic_node)
builder.add_node("research_manager", research_manager_node)
builder.add_node("content_manager", content_manager_node)
builder.add_node("research_execution", research_execution_node)
builder.add_node("content_execution", content_execution_node)
builder.add_node("final_assembly", final_assembly_node)

builder.set_entry_point("strategic")
builder.add_edge("strategic", "research_manager")
builder.add_edge("research_manager", "research_execution")
builder.add_edge("research_execution", "content_manager")
builder.add_edge("content_manager", "content_execution")
builder.add_edge("content_execution", "final_assembly")
builder.add_edge("final_assembly", END)

graph = builder.compile()
print("Hierarchical agent graph compiled.")

## Live Demo

In [ ]:
goal = (
    "Produce a one-page briefing document on the impact of AI on software engineering jobs "
    "in 2025. Include current data, key concerns, and a balanced conclusion. "
    "Target audience: non-technical business executives."
)

result = graph.invoke({
    "goal": goal, "research_objective": "", "content_objective": "",
    "research_tasks": [], "content_tasks": [],
    "research_results": [], "content_pieces": [], "final_output": None
})

print("="*60)
print("FINAL DELIVERABLE")
print("="*60)
print(result["final_output"])

## Key Takeaways

| Concept | Implementation |
|---------|---------------|
| **3-tier hierarchy** | Strategic sets goals → Tactical plans tasks → Execution does work |
| **Information flows down** | Each tier passes refined context to the next |
| **Isolation** | Workers only need to understand their task |
| **Scalability** | Add departments without redesigning upper layers |

## What's Next

**Notebook 12** closes the loop with LLM-as-a-Judge evaluation.